# Hard Thresholding long lower-rank search on Fritz

The refined search selected its lowest tested rank and its last iteration. This notebook searches lower ranks with substantially longer trajectories.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse.linalg import ArpackNoConvergence
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hard_thresholding_hpc import (
    hard_thresholding_model_matrix_free,
    predict_pairs,
)

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml-latest"
REFINED_RESULTS_DIR = PROJECT_ROOT / "results" / "hard_thresholding_refined"
RESULTS_DIR = PROJECT_ROOT / "results" / "hard_thresholding_long"
HISTORIES_DIR = RESULTS_DIR / "histories"
HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_ratings.csv")
user_map = pd.read_csv(PROCESSED_DIR / "user_map.csv")
movie_map = pd.read_csv(PROCESSED_DIR / "movie_map.csv")

n_users = len(user_map)
n_movies = len(movie_map)

train_fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

val_user_idx = val_df["user_idx"].to_numpy(dtype=np.int64)
val_movie_idx = val_df["movie_idx"].to_numpy(dtype=np.int64)
val_ratings = val_df["rating"].to_numpy(dtype=np.float64)

print("Train fit shape:", train_fit_df.shape)
print("Validation shape:", val_df.shape)
print("Number of users:", n_users)
print("Number of movies:", n_movies)

In [ ]:
refined_results_path = REFINED_RESULTS_DIR / "best_by_rank.csv"
if not refined_results_path.exists():
    raise FileNotFoundError(
        "Run the refined-search Slurm job before the long search: "
        f"{refined_results_path}"
    )

refined_best_by_rank = pd.read_csv(refined_results_path)
print("Previous refined-search results:")
display(refined_best_by_rank)

In [ ]:
def hard_validation_metrics(U, singular_values, V):
    predictions = predict_pairs(
        U,
        singular_values,
        V,
        val_user_idx,
        val_movie_idx,
    )
    predictions = np.clip(predictions, 0.5, 5.0)
    return {
        "val_rmse": float(np.sqrt(np.mean((val_ratings - predictions) ** 2))),
    }

## Long lower-rank grid

Ranks 8-24 test the region below the previous boundary winner, rank 20. Every trajectory is long enough to compare ranks closer to convergence.

In [ ]:
long_rank_values = [8, 12, 16, 20, 24]
long_grid_max_iter = 180
finalist_max_iter = 300
hard_svd_tol = 1e-4
hard_svd_maxiter = 300

print("Long-search ranks:", long_rank_values)
print("Iterations per rank:", long_grid_max_iter)
print("Boundary finalist iterations:", finalist_max_iter)

## Long search

In [ ]:
# Compile the Numba prediction kernel before the timed search.
predict_pairs(
    np.ones((2, 1)),
    np.ones(1),
    np.ones((2, 1)),
    np.array([0], dtype=np.int64),
    np.array([0], dtype=np.int64),
)

long_histories = []
long_failed_configs = []
long_search_start = perf_counter()

for config_number, rank in enumerate(long_rank_values, start=1):
    history_path = HISTORIES_DIR / f"rank-{rank}.csv"
    print(f"[{config_number}/{len(long_rank_values)}] rank={rank}")

    if history_path.exists():
        print("  using saved history")
        long_histories.append(pd.read_csv(history_path))
        continue

    config_start = perf_counter()
    try:
        _, _, _, history_df = hard_thresholding_model_matrix_free(
            train_fit_df,
            n_users=n_users,
            n_movies=n_movies,
            rank=rank,
            n_iter=long_grid_max_iter,
            metrics_fn=hard_validation_metrics,
            svd_tol=hard_svd_tol,
            svd_maxiter=hard_svd_maxiter,
            random_state=42,
        )
    except ArpackNoConvergence as error:
        long_failed_configs.append({
            "rank": rank,
            "error": repr(error),
        })
        print("  failed to converge:", repr(error))
        continue

    config_seconds = perf_counter() - config_start
    history_df["config_seconds"] = config_seconds
    history_df.to_csv(history_path, index=False)
    long_histories.append(history_df)
    print(
        f"  {config_seconds / 60:.1f} min; "
        f"best iteration={int(history_df.loc[history_df['val_rmse'].idxmin(), 'iteration'])}; "
        f"best val RMSE={history_df['val_rmse'].min():.6f}"
    )

if not long_histories:
    raise RuntimeError("Every long-search hard-thresholding configuration failed")

long_results_df = pd.concat(long_histories, ignore_index=True)
long_best_by_rank = (
    long_results_df.loc[
        long_results_df.groupby("rank")["val_rmse"].idxmin()
    ]
    .sort_values("val_rmse")
    .reset_index(drop=True)
)
long_results_df.to_csv(RESULTS_DIR / "all_histories.csv", index=False)
long_best_by_rank.to_csv(RESULTS_DIR / "best_by_rank.csv", index=False)

if long_failed_configs:
    pd.DataFrame(long_failed_configs).to_csv(
        RESULTS_DIR / "failed_configs.csv",
        index=False,
    )

print(f"Long search time: {(perf_counter() - long_search_start) / 60:.1f} min")
display(long_best_by_rank)

## Select the best rank and retained iteration

In [ ]:
best_long_row = long_best_by_rank.iloc[0]
best_hard_rank = int(best_long_row["rank"])
best_long_iteration = int(best_long_row["iteration"])

if best_long_iteration == long_grid_max_iter:
    print(
        f"Best rank={best_hard_rank} is still best at iteration {long_grid_max_iter}; "
        f"extending it to {finalist_max_iter} iterations..."
    )
    _, _, _, selected_history = hard_thresholding_model_matrix_free(
        train_fit_df,
        n_users=n_users,
        n_movies=n_movies,
        rank=best_hard_rank,
        n_iter=finalist_max_iter,
        metrics_fn=hard_validation_metrics,
        svd_tol=hard_svd_tol,
        svd_maxiter=hard_svd_maxiter,
        random_state=42,
    )
else:
    selected_history = (
        long_results_df[long_results_df["rank"] == best_hard_rank]
        .sort_values("iteration")
        .reset_index(drop=True)
    )

best_iteration_row = selected_history.loc[
    selected_history["val_rmse"].idxmin()
]
best_hard_n_iter = int(best_iteration_row["iteration"])
best_hard_val_rmse = float(best_iteration_row["val_rmse"])

best_parameters_df = pd.DataFrame([{
    "rank": best_hard_rank,
    "n_iter": best_hard_n_iter,
    "val_rmse": best_hard_val_rmse,
}])
selected_history.to_csv(RESULTS_DIR / "selected_history.csv", index=False)
best_parameters_df.to_csv(RESULTS_DIR / "best_parameters.csv", index=False)
display(best_parameters_df)

if best_hard_n_iter == int(selected_history["iteration"].max()):
    print("Warning: validation RMSE is still best at the final iteration.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(selected_history["iteration"], selected_history["train_rmse"], label="Train RMSE")
plt.plot(selected_history["iteration"], selected_history["val_rmse"], label="Validation RMSE")
plt.axvline(
    best_hard_n_iter,
    linestyle="--",
    color="black",
    label=f"Retained iteration = {best_hard_n_iter}",
)
plt.xlabel("Iteration")
plt.ylabel("RMSE")
plt.title(f"Hard Thresholding: long trajectory (rank={best_hard_rank})")
plt.legend()
plt.grid(alpha=0.3)
plt.show()